# Citadel Colab TPU launcher (thin — first execution surface)
All logic lives in `citadel_tpu/`. This notebook only: sync latest `citadel` → setup PJRT → probe → T0 → STOP unless T0 passes → export receipts. No secrets are used or printed.

In [ ]:
# 0. Obtain/sync repo at latest citadel (public clone, no credentials)
import os, subprocess
repo = '/content/An-Ra-colab'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git','clone','--depth','50','-b','citadel','https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git',repo], check=True)
else:
    subprocess.run(['git','-C',repo,'fetch','origin','citadel','--depth','50'], check=True)
    subprocess.run(['git','-C',repo,'checkout','citadel'], check=True)
    subprocess.run(['git','-C',repo,'reset','--hard','origin/citadel'], check=True)
%cd /content/An-Ra-colab
!git rev-parse HEAD && git log -1 --oneline

In [ ]:
# 1. Inspect the handover (authoritative next-action source)
!sed -n '1,100p' agent.md

In [ ]:
# 2. Select PJRT TPU before importing torch-xla; inspect versions first
import os
os.environ.setdefault('PJRT_DEVICE', 'TPU')
os.environ['CITADEL_PLATFORM'] = 'colab'
print('PJRT_DEVICE', os.environ.get('PJRT_DEVICE'))
!python -c "import torch; print('torch', torch.__version__)" 2>&1 | tail -1
!python -c "import torch_xla; print('torch-xla', getattr(torch_xla, '__version__', 'unknown'))" 2>&1 | tail -1
!python -c "import numpy; print('numpy', numpy.__version__)" 2>&1 | tail -1

In [ ]:
# 3. Minimal conditional install (run ONLY if cell 2 showed a missing package)
# Do not reinstall a working torch/torch-xla pair.
# !pip install -q torch torch-xla numpy 2>&1 | tail -2

In [ ]:
# 4. M0: environment probe (PJRT-aware, fail-closed on CPU fallback)
from citadel_tpu import environment as env_mod
env = env_mod.main(out='docs/citadel/tpu_receipts/TPU_ENVIRONMENT.json', require_tpu=True, platform_override='colab')
print({k: env[k] for k in ('platform','accelerator_detected','xla_device_count','torch_version','torch_xla_version','probe_pass')})

In [ ]:
# 5. T0: single-device one-update certification (MINI_SPEC, bucket 512, CE, one update)
from citadel_tpu import one_update
r0 = one_update.run(out='docs/citadel/tpu_receipts/TPU_ONE_UPDATE.json')
print({k: r0[k] for k in ('certification','loss','tokens_per_second','reload_identical')})

In [ ]:
# 6. STOP unless T0 passed. Export exact receipt files for operator transfer.
assert r0.get('certification') == 'PASS', 'T0 did not pass — STOP. Diagnose, do not escalate.'
from google.colab import files
files.download('docs/citadel/tpu_receipts/TPU_ENVIRONMENT.json')
files.download('docs/citadel/tpu_receipts/TPU_ONE_UPDATE.json')
print('exported; transfer these exact files back to the operator')